# Experiment 6 — DBSCAN clustering

## Run directly in Jupyter

- Keep the extracted `datasets` folder beside the `notebooks` folder.
- Run every cell from top to bottom with **Shift + Enter**, or choose **Run → Run All Cells**.
- No manual input or internet download is required.
- If imports are missing, run `%pip install numpy pandas matplotlib seaborn scikit-learn` in a code cell, then rerun the failed cell.

**Dataset:** `datasets/sample_titanic.csv` is an included synthetic teaching dataset, not the official Titanic passenger dataset. Scores are illustrative.

In [ ]:
# Load the included sample dataset. No internet or manual input is needed.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

possible_paths = [
    Path("sample_titanic.csv"),
    Path("../datasets/sample_titanic.csv"),
    Path("datasets/sample_titanic.csv"),
]
csv_file = next((p for p in possible_paths if p.exists()), None)
if csv_file is None:
    raise FileNotFoundError(
        "sample_titanic.csv not found. Keep the datasets folder beside notebooks."
    )

df = pd.read_csv(csv_file)
df = df.rename(columns={
    "PassengerId": "passenger_id", "Survived": "survived", "Pclass": "pclass",
    "Name": "name", "Sex": "sex", "Age": "age", "SibSp": "sibsp",
    "Parch": "parch", "Fare": "fare", "Embarked": "embarked", "Cabin": "cabin"
})
print("Loaded sample dataset:", csv_file)
print("Dataset shape:", df.shape)
display(df.head())


## Aim
Cluster numeric Titanic passenger features using DBSCAN. Label `-1` means noise.

## 1. Select and scale features

In [ ]:
features=["pclass","age","sibsp","parch","fare"]
X=df[features].copy()
for col in features: X[col]=X[col].fillna(X[col].median())
from sklearn.preprocessing import StandardScaler
X_scaled=StandardScaler().fit_transform(X)
print("Shape:",X.shape)
print("Missing values:\n",X.isnull().sum())


## 2. Fit DBSCAN and inspect labels

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
labels=DBSCAN(eps=0.8,min_samples=8).fit_predict(X_scaled)
df["Cluster"]=labels
n_clusters=len(set(labels))-int(-1 in labels)
n_noise=int(np.sum(labels==-1))
print("Cluster counts (label -1 means noise):\n",pd.Series(labels).value_counts().sort_index())
print("Number of clusters:",n_clusters)
print("Number of noise points:",n_noise)
mask=labels!=-1
if len(set(labels[mask]))>1:
    print("Silhouette score excluding noise:",round(silhouette_score(X_scaled[mask],labels[mask]),3))
else:
    print("Silhouette score skipped: fewer than two non-noise clusters.")


## 3. Plot clusters

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(X["age"],X["fare"],c=labels,cmap="viridis",s=25)
plt.xlabel("Age"); plt.ylabel("Fare"); plt.title("DBSCAN Clustering — Titanic Features")
plt.colorbar(label="Cluster label (-1 = noise)"); plt.show()


## Expected output
Cluster counts, number of clusters/noise points, optional silhouette score and a scatter plot. DBSCAN results depend on eps and min_samples.